In [ ]:
# ── CELL 1: Imports & Setup ──────────────────────────────────────────────────
import os
import json
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

print("TensorFlow version:", tf.__version__)

dataset_path = r"E:\Files\Downloads\PlantDiseaseProject\dataset\Plant_leave_diseases_dataset_with_augmentation"
print("Dataset folders (first 5):", os.listdir(dataset_path)[:5])

In [3]:
# ── CELL 2: Load Dataset ─────────────────────────────────────────────────────
IMG_SIZE   = (160, 160)
BATCH_SIZE = 32
 
train_dataset = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)
 
validation_dataset = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)
 
class_names = train_dataset.class_names
num_classes = len(class_names)
 
print(f"Total Classes: {num_classes}")
print("Classes:", class_names)
 
# Save class names immediately
os.makedirs("models", exist_ok=True)
with open("models/class_names.json", "w") as f:
    json.dump(class_names, f)
print("✅ class_names.json saved!")
 

Found 61486 files belonging to 39 classes.
Using 49189 files for training.
Found 61486 files belonging to 39 classes.
Using 12297 files for validation.
Total Classes: 39
Classes: ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Background_without_leaves', 'Blueberry___healthy', 'Cherry___Powdery_mildew', 'Cherry___healthy', 'Corn___Cercospora_leaf_spot Gray_leaf_spot', 'Corn___Common_rust', 'Corn___Northern_Leaf_Blight', 'Corn___healthy', 'Grape___Black_rot', 'Grape___Esca_(Black_Measles)', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Grape___healthy', 'Orange___Haunglongbing_(Citrus_greening)', 'Peach___Bacterial_spot', 'Peach___healthy', 'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Raspberry___healthy', 'Soybean___healthy', 'Squash___Powdery_mildew', 'Strawberry___Leaf_scorch', 'Strawberry___healthy', 'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tom

In [4]:
# ── CELL 3: Optimize Dataset Pipeline ───────────────────────────────────────
AUTOTUNE = tf.data.AUTOTUNE
train_dataset      = train_dataset.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
validation_dataset = validation_dataset.cache().prefetch(buffer_size=AUTOTUNE)

In [5]:
# ── CELL 4: Visualize Sample Images ─────────────────────────────────────────
plt.figure(figsize=(10, 10))
for images, labels in train_dataset.take(1):
    for i in range(9):
        plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[labels[i]])
        plt.axis("off")
plt.suptitle("Sample Training Images")
plt.tight_layout()
plt.show()

: 

In [ ]:
# ── CELL 5: Data Augmentation Layer (used ONLY during training) ──────────────
# ✅ FIX Bug 2: Augmentation is defined separately and applied manually,
#    NOT baked inside the model. This means inference is clean/deterministic.
 
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.3),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.2),
    layers.RandomBrightness(0.2),
], name="data_augmentation")
 

In [ ]:
# ── CELL 6: Augmented Training Datasets ─────────────────────────────────────
# Apply augmentation only to training data, not validation
def augment(image, label):
    image = data_augmentation(image, training=True)
    return image, label

train_dataset_aug = train_dataset.map(augment, num_parallel_calls=AUTOTUNE)

In [ ]:
# ── CELL 7: MODEL 1 — Basic CNN (Fixed) ─────────────────────────────────────
# 
# 
 
basic_cnn = models.Sequential([
    layers.Input(shape=(160, 160, 3)),
    layers.Rescaling(1./255),           # Normalize to [0, 1]
 
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),
 
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),
 
    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),
 
    layers.Conv2D(256, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),
 
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation='softmax')
], name="basic_cnn")
 
basic_cnn.summary()
 
basic_cnn.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
 
# Callbacks for Basic CNN
callbacks_basic = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        "models/basic_cnn_best.keras", monitor='val_accuracy',
        save_best_only=True, verbose=1
    )
]
 
print("\n🚀 Training Basic CNN on FULL dataset for up to 25 epochs...")
history_basic = basic_cnn.fit(
    train_dataset_aug,                  # ✅ full augmented dataset
    validation_data=validation_dataset,
    epochs=25,                          # 
    callbacks=callbacks_basic
)
 
basic_cnn.save("models/basic_cnn_model.keras")
print("✅ Basic CNN saved!")
 

In [ ]:
# ── CELL 8: Plot Basic CNN Training ─────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
 
ax1.plot(history_basic.history['accuracy'],     label='Train Accuracy')
ax1.plot(history_basic.history['val_accuracy'], label='Val Accuracy')
ax1.set_title("Basic CNN — Accuracy")
ax1.set_xlabel("Epoch")
ax1.legend()
 
ax2.plot(history_basic.history['loss'],     label='Train Loss')
ax2.plot(history_basic.history['val_loss'], label='Val Loss')
ax2.set_title("Basic CNN — Loss")
ax2.set_xlabel("Epoch")
ax2.legend()
 
plt.tight_layout()
plt.show()
 
print(f"Best Basic CNN Val Accuracy: {max(history_basic.history['val_accuracy'])*100:.2f}%")

In [ ]:
# ── CELL 9: MODEL 2 — MobileNetV2 Base Training ──────────────────────────────
# ✅ FIX Bug 1: Use preprocess_input (scales to [-1, 1]) NOT Rescaling(1./255)
# ✅ FIX Bug 2: No augmentation inside model
# ✅ FIX Bug 4: 25 epochs base training
 
base_model = MobileNetV2(
    input_shape=(160, 160, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False  # Freeze for base training
 
# ✅ Correct preprocessing pipeline for MobileNetV2
# preprocess_input scales pixels to [-1, 1] which is what MobileNetV2 expects
def preprocess_mobilenet(image, label):
    image = preprocess_input(image)   # ✅ Bug 1 fix: correct scaling
    return image, label
 
train_dataset_mob      = train_dataset_aug.map(
    lambda x, y: preprocess_mobilenet(x, y), num_parallel_calls=AUTOTUNE
)
validation_dataset_mob = validation_dataset.map(
    lambda x, y: preprocess_mobilenet(x, y), num_parallel_calls=AUTOTUNE
)
 
mobilenet_model = models.Sequential([
    layers.Input(shape=(160, 160, 3)),
    # ✅ NO augmentation layer here — applied in dataset pipeline above
    # ✅ NO Rescaling layer — preprocess_input handles it
 
    base_model,
 
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation='softmax')
], name="mobilenetv2")

mobilenet_model.summary()
print(f"\nTrainable params (base training): {mobilenet_model.count_params():,}")
 
mobilenet_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_mob = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        "models/mobilenetv2_base_best.keras", monitor='val_accuracy',
        save_best_only=True, verbose=1
    )
]
 
print("\n🚀 Training MobileNetV2 (frozen base) for up to 25 epochs...")
history_mob = mobilenet_model.fit(
    train_dataset_mob,
    validation_data=validation_dataset_mob,
    epochs=25,                              # ✅ Bug 4 fix
    callbacks=callbacks_mob
)
 
print(f"Best Base Training Val Accuracy: {max(history_mob.history['val_accuracy'])*100:.2f}%")

In [ ]:
# ── CELL 10: Plot MobileNetV2 Base Training ──────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
 
ax1.plot(history_mob.history['accuracy'],     label='Train Accuracy')
ax1.plot(history_mob.history['val_accuracy'], label='Val Accuracy')
ax1.set_title("MobileNetV2 Base Training — Accuracy")
ax1.set_xlabel("Epoch")
ax1.legend()

ax2.plot(history_mob.history['loss'],     label='Train Loss')
ax2.plot(history_mob.history['val_loss'], label='Val Loss')
ax2.set_title("MobileNetV2 Base Training — Loss")
ax2.set_xlabel("Epoch")
ax2.legend()
 
plt.tight_layout()
plt.show()
 

In [ ]:
 
# ── CELL 11: Fine-Tuning MobileNetV2 ────────────────────────────────────────
# Unfreeze last 50 layers for deeper fine-tuning (was 30, now 50)
base_model.trainable = True
 
for layer in base_model.layers[:-50]:   # ✅ Unfreeze last 50 layers
    layer.trainable = False
 
trainable_count = sum(1 for l in base_model.layers if l.trainable)
print(f"Trainable layers in base_model: {trainable_count} / {len(base_model.layers)}")
 
# Much lower learning rate for fine-tuning to avoid destroying pretrained weights
mobilenet_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),  # Very low LR
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
 
callbacks_finetune = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=7, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.3, patience=3, verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        "models/mobilenetv2_finetuned_best.keras", monitor='val_accuracy',
        save_best_only=True, verbose=1
    )
]
 
print("\n🔥 Fine-tuning MobileNetV2 (last 50 layers unfrozen) for up to 10 epochs...")
history_fine = mobilenet_model.fit(
    train_dataset_mob,
    validation_data=validation_dataset_mob,
    epochs=10,                              # ✅ Bug 4 fix: was 3, now 10
    callbacks=callbacks_finetune
)
 
mobilenet_model.save("models/mobilenetv2_finetuned.keras")
print("✅ Fine-tuned MobileNetV2 saved!")
print(f"Best Fine-Tuned Val Accuracy: {max(history_fine.history['val_accuracy'])*100:.2f}%")

In [ ]:
# ── CELL 12: Fine-Tuning Comparison Plot ────────────────────────────────────
plt.figure(figsize=(10, 5))
plt.plot(history_mob.history['val_accuracy'],  label='Before Fine-Tuning', linewidth=2)
plt.plot(history_fine.history['val_accuracy'], label='After Fine-Tuning',  linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.title("MobileNetV2 — Fine-Tuning Comparison")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# ── CELL 13: Full Model Evaluation ──────────────────────────────────────────
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
 
# Evaluate on validation set (same preprocessing)
y_true = np.concatenate([y for _, y in validation_dataset_mob], axis=0)
y_pred_probs = mobilenet_model.predict(validation_dataset_mob)
y_pred = np.argmax(y_pred_probs, axis=1)
 
# Classification Report
print("\n📊 Classification Report (MobileNetV2 Fine-tuned):")
print(classification_report(y_true, y_pred, target_names=class_names))
 
# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(20, 16))
sns.heatmap(cm, annot=False, cmap="Blues",
            xticklabels=class_names,
            yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix — MobileNetV2 (Fine-Tuned)")
plt.xticks(rotation=90, fontsize=7)
plt.yticks(rotation=0,  fontsize=7)
plt.tight_layout()
plt.show()
 

In [ ]:
# ── CELL 14: Accuracy Comparison Table ──────────────────────────────────────
import pandas as pd
 
comparison = pd.DataFrame({
    "Model": [
        "Basic CNN (Full Dataset)",
        "MobileNetV2 (Before Fine-Tuning)",
        "MobileNetV2 (After Fine-Tuning)"
    ],
    "Best Val Accuracy (%)": [
        round(max(history_basic.history['val_accuracy']) * 100, 2),
        round(max(history_mob.history['val_accuracy'])   * 100, 2),
        round(max(history_fine.history['val_accuracy'])  * 100, 2),
    ]
})
 
print("\n" + "=" * 55)
print(comparison.to_string(index=False))
print("=" * 55)

In [ ]:
# ── CELL 15: Parameter Count Comparison ─────────────────────────────────────
print(f"\nBasic CNN Parameters    : {basic_cnn.count_params():,}")
print(f"MobileNetV2 Parameters  : {mobilenet_model.count_params():,}")

In [ ]:
# ── CELL 16: Overfitting Analysis ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
 
axes[0].plot(history_fine.history['accuracy'],     label='Train Accuracy', linewidth=2)
axes[0].plot(history_fine.history['val_accuracy'], label='Val Accuracy',   linewidth=2)
axes[0].set_title("Overfitting Analysis — MobileNetV2 Fine-Tuning")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[0].grid(True)
 
axes[1].plot(history_fine.history['loss'],     label='Train Loss', linewidth=2)
axes[1].plot(history_fine.history['val_loss'], label='Val Loss',   linewidth=2)
axes[1].set_title("Loss Curve — MobileNetV2 Fine-Tuning")
axes[1].set_xlabel("Epoch")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# ── CELL 17: Single Image Real-World Test ────────────────────────────────────
# ✅ IMPORTANT: This uses correct preprocessing (preprocess_input)
from tensorflow.keras.preprocessing import image as keras_image
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
 
img_path = "test_images/testleaf.jpg"   # Change to your test image path
 
img       = keras_image.load_img(img_path, target_size=(160, 160))
img_array = keras_image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)
img_array = preprocess_input(img_array)        # ✅ Correct preprocessing
 
prediction       = mobilenet_model.predict(img_array)
predicted_idx    = np.argmax(prediction)
predicted_class  = class_names[predicted_idx]
confidence       = prediction[0][predicted_idx] * 100
 
print(f"Prediction  : {predicted_class}")
print(f"Confidence  : {confidence:.2f}%")
 
plt.imshow(keras_image.load_img(img_path, target_size=(160, 160)))
plt.title(f"Prediction: {predicted_class}\nConfidence: {confidence:.2f}%")
plt.axis("off")
plt.show()

In [ ]:
 
# ── CELL 18: Summary ─────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  TRAINING COMPLETE — FILES SAVED")
print("=" * 60)
print("  models/basic_cnn_model.keras")
print("  models/mobilenetv2_finetuned.keras")
print("  models/class_names.json")
print("=" * 60)
print(f"\n  Basic CNN Val Accuracy      : {max(history_basic.history['val_accuracy'])*100:.2f}%")
print(f"  MobileNetV2 Base Accuracy   : {max(history_mob.history['val_accuracy'])*100:.2f}%")
print(f"  MobileNetV2 Fine-Tuned Acc  : {max(history_fine.history['val_accuracy'])*100:.2f}%")
print("=" * 60)
print("\n  ✅ Run test_realworld.py next to check real-world accuracy!")